In [1]:
# Import statements
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from statsmodels.tsa.stattools import acf
import epyestim.covid19 as c19
from epyestim.distributions import discretise_gamma

In [2]:
cases = pd.read_csv('cases_total.csv')
cases['date'] = pd.to_datetime(cases['date'])
cases.set_index('date',inplace=True)
npis = pd.read_csv('ltla_to_nhs.csv')

cases_region = pd.merge(cases.reset_index(), npis, left_on='area_name', right_on='ltla20nm')
cases_region.drop(columns=['ltla20nm','area_code','area_name','Unnamed: 0'],inplace=True)
cases_region = cases_region.groupby(['date','region']).sum().reset_index().set_index('date')
cases_region

,region,value
date,,
2020-06-01,East of England,142.0
2020-06-01,London,50.0
2020-06-01,Midlands,332.0
2020-06-01,North East and Yorkshire,273.0
2020-06-01,North West,262.0
...,...,...
2021-12-31,Midlands,20460.0
2021-12-31,North East and Yorkshire,20634.0
2021-12-31,North West,19527.0


In [3]:
# areas = np.unique(cases['area_name'].values)
def find_critical_transitions_nhs(timeseries):
    fitted_r = c19.r_covid(timeseries,smoothing_window=35,gt_distribution=discretise_gamma(8.5,0.62))
    crossing_days = []
    values = fitted_r['R_mean'].values
    # dates = timeseries.reset_index()['date'].values
    dates = fitted_r.reset_index()['index'].values
    # Loop through the values to find transitions through 1
    for i in range(len(values) - 1):
        if (values[i] < 1 and values[i+1] > 1) or (values[i] > 1 and values[i+1] < 1):
            crossing_days.append(dates[i+1])
    crossing_days.append(pd.Timestamp.max)
    return crossing_days

areas = np.unique(cases_region['region'].values)
nhs_transitions = {}
for area in areas:
    timeseries = cases_region[cases_region['region']==area]['value']
    nhs_transitions[area] = find_critical_transitions_nhs(timeseries)

In [4]:
cases_national = cases_region.reset_index().drop(columns='region').groupby('date').sum()

def find_critical_transitions_national(timeseries):
    fitted_r = c19.r_covid(timeseries,smoothing_window=35,gt_distribution=discretise_gamma(8.5,0.62))
    crossing_days = []
    values = fitted_r['R_mean'].values
    # dates = timeseries.reset_index()['date'].values
    dates = fitted_r.reset_index()['index'].values
    # Loop through the values to find transitions through 1
    for i in range(len(values) - 1):
        if (values[i] < 1 and values[i+1] > 1) or (values[i] > 1 and values[i+1] < 1):
            crossing_days.append(dates[i+1])
    crossing_days.append(pd.Timestamp.max)
    return crossing_days

national_transitions = find_critical_transitions_national(cases_national['value'])

In [6]:
import pickle
with open('nhs_transitions.pkl','wb') as f:
    pickle.dump(nhs_transitions,f)

with open('england_transitions.npy','wb') as f:
    np.save(f, national_transitions)